In [1]:
"""
1. 데이터 불러오기
2. downsampling -> 간단하게 interpolation
3. Statistical Features 계산 -> total_acc_x/z mean, 각속도_z min/max
4. Z-score normalization
5. Multi-output CNN
6. CNN 학습
7. Routing label 생성
   - FOB 맞음, Baseline 맞음 → 1
   - FOB 맞음, Baseline 틀림 → 1
   - FOB 틀림, Baseline 맞음 → 2
   - FOB 틀림, Baseline 틀림 → 1
8. DT 학습
9. Inference
10. 로그출력: CNN 학습 로그, DT 학습 결과, FOB/Baseline/Proposed 결과(Macro-F1만)
"""

import copy
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report


# ==============================================================================
# 1. Load Data
# ==============================================================================
CLASS_NAMES = ["WALK", "UP", "DOWN", "SIT", "STAND", "LAY"]
class UCIHARDataset(Dataset):
    def __init__(self, data_dir, split="train", normalize=None):
        self.data_dir = Path(data_dir)
        self.split    = split
        self.X, self.y = self._load_data()
        self.X = torch.FloatTensor(self.X)
        self.y = torch.LongTensor(self.y) - 1

        self.normalize = normalize

    def _load_data(self):
        split_dir    = self.data_dir / self.split
        signal_types = [
            "body_acc_x","body_acc_y","body_acc_z",
            "body_gyro_x","body_gyro_y","body_gyro_z",
            "total_acc_x","total_acc_y","total_acc_z",
        ]
        signals = []
        for st in signal_types:
            fname = split_dir / "Inertial Signals" / f"{st}_{self.split}.txt"
            signals.append(np.loadtxt(fname))
        X = np.stack(signals, axis=1)
        y = np.loadtxt(split_dir / f"y_{self.split}.txt", dtype=int)
        return X, y

    def __len__(self):  return len(self.X)

    def __getitem__(self, idx):
        X = self.X[idx]
        y = self.y[idx]
        if self.normalize is not None:
            mean, std = self.normalize
            X = (X - mean.squeeze(0)) / std.squeeze(0)
        return X, y


# ==============================================================================
# 2. Downsampling: 128 → 32 using interpolation
# ==============================================================================
def downsample_interpolation(X, target_len=32):
    X_down = F.interpolate(
        X,
        size=target_len,
        mode="linear",
        align_corners=False
    )
    return X_down


# ==============================================================================
# 3. Statistical Features for OBP
# ==============================================================================
def extract_ahar_stat_features(X):
    if isinstance(X, torch.Tensor):
        X_np = X.detach().cpu().numpy()
    else:
        X_np = X

    total_acc_x = X_np[:, 6, :]
    total_acc_z = X_np[:, 8, :]
    gyro_z      = X_np[:, 5, :]

    return np.stack([
        total_acc_x.mean(axis=1),
        total_acc_z.mean(axis=1),
        gyro_z.min(axis=1),
        gyro_z.max(axis=1),
    ], axis=1)


# ==============================================================================
# 4. Segment-wise Z-score Normalization
# ==============================================================================
def segment_wise_zscore(X, eps=1e-8):
    mean = X.mean(dim=2, keepdim=True)
    std  = X.std(dim=2, keepdim=True)

    X_norm = (X - mean) / (std + eps)

    return X_norm


# ==============================================================================
# 5. Multi-output CNN (TABLE IV)
# ==============================================================================
class MultiOutputCNN(nn.Module):
    def __init__(self, in_channels=9, num_classes=6):
        super().__init__()

        # First Conv Block: [Conv1D → LeakyReLU → AvgPooling → BatchNorm]
        self.conv_block1 = nn.Sequential(
            nn.Conv1d(
                in_channels=in_channels,
                out_channels=6,
                kernel_size=5,
                stride=3
            ),
            nn.LeakyReLU(),
            nn.AvgPool1d(
                kernel_size=2,
                stride=2
            ),
            nn.BatchNorm1d(6)
        )

        self.fob_classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(5 * 6, num_classes)
        )

        # Second Conv Block: [Conv1D → LeakyReLU → BatchNorm]
        self.conv_block2 = nn.Sequential(
            nn.Conv1d(
                in_channels=6,
                out_channels=8,
                kernel_size=4,
                stride=1
            ),
            nn.LeakyReLU(),
            nn.BatchNorm1d(8)
        )

        self.baseline_classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2 * 8, 16),
            nn.LeakyReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        h1 = self.conv_block1(x)

        # First Output Block
        fob_logits = self.fob_classifier(h1)

        # Second Output Block / Baseline
        h2 = self.conv_block2(h1)
        baseline_logits = self.baseline_classifier(h2)

        return fob_logits, baseline_logits

    def forward_fob(self, x):
        h1 = self.conv_block1(x)
        return self.fob_classifier(h1)

    def forward_baseline(self, x):
        h1 = self.conv_block1(x)
        h2 = self.conv_block2(h1)
        return self.baseline_classifier(h2)

# ==============================================================================
# 6. Train & Eval CNN
# ==============================================================================
def evaluate_multioutput_cnn(model, loader, device):
    model.eval()

    all_y = []
    all_fob_pred = []
    all_base_pred = []

    total_loss = 0.0
    ce = nn.CrossEntropyLoss()

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            y = y.to(device)

            fob_logits, base_logits = model(X)

            loss = ce(fob_logits, y) + ce(base_logits, y)
            total_loss += loss.item() * X.size(0)

            fob_pred = fob_logits.argmax(dim=1)
            base_pred = base_logits.argmax(dim=1)

            all_y.append(y.cpu().numpy())
            all_fob_pred.append(fob_pred.cpu().numpy())
            all_base_pred.append(base_pred.cpu().numpy())

    all_y = np.concatenate(all_y)
    all_fob_pred = np.concatenate(all_fob_pred)
    all_base_pred = np.concatenate(all_base_pred)

    return {
        "loss": total_loss / len(loader.dataset),

        "fob_acc": accuracy_score(all_y, all_fob_pred),
        "fob_macro_f1": f1_score(all_y, all_fob_pred, average="macro"),

        "base_acc": accuracy_score(all_y, all_base_pred),
        "base_macro_f1": f1_score(all_y, all_base_pred, average="macro"),
    }


def train_multioutput_cnn(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    batch_size=64,
    epochs=100,
    lr=0.007,
    weight_decay=0.0,
    device=None
):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    test_loader = DataLoader(
        TensorDataset(X_test, y_test),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    model = model.to(device)
    ce = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_state = copy.deepcopy(model.state_dict())
    best_base_f1 = -1.0

    history = []

    for epoch in range(1, epochs + 1):
        model.train()

        train_loss = 0.0
        all_y = []
        all_fob_pred = []
        all_base_pred = []

        for X, y in train_loader:
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            fob_logits, base_logits = model(X)

            loss_fob = ce(fob_logits, y)
            loss_base = ce(base_logits, y)
            loss = loss_fob + loss_base

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X.size(0)

            all_y.append(y.detach().cpu().numpy())
            all_fob_pred.append(fob_logits.argmax(dim=1).detach().cpu().numpy())
            all_base_pred.append(base_logits.argmax(dim=1).detach().cpu().numpy())

        all_y = np.concatenate(all_y)
        all_fob_pred = np.concatenate(all_fob_pred)
        all_base_pred = np.concatenate(all_base_pred)

        train_metrics = {
            "loss": train_loss / len(train_loader.dataset),
            "fob_acc": accuracy_score(all_y, all_fob_pred),
            "fob_macro_f1": f1_score(all_y, all_fob_pred, average="macro"),
            "base_acc": accuracy_score(all_y, all_base_pred),
            "base_macro_f1": f1_score(all_y, all_base_pred, average="macro"),
        }

        test_metrics = evaluate_multioutput_cnn(
            model=model,
            loader=test_loader,
            device=device
        )

        if test_metrics["base_macro_f1"] > best_base_f1:
            best_base_f1 = test_metrics["base_macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            best_mark = " BEST"
        else:
            best_mark = ""

        history.append({
            "epoch": epoch,
            "train": train_metrics,
            "test": test_metrics
        })

        if (epoch % 25 == 0) or (epoch == 1) or (epoch == epochs):
            print(
                f"[Epoch {epoch:03d}/{epochs}] "
                f"Train Loss={train_metrics['loss']:.4f} | "
                f"Train FOB F1={train_metrics['fob_macro_f1']:.4f}, "
                f"Base F1={train_metrics['base_macro_f1']:.4f} || "
                f"Test FOB F1={test_metrics['fob_macro_f1']:.4f}, "
                f"Base F1={test_metrics['base_macro_f1']:.4f}"
                f"{best_mark}"
            )
    model.load_state_dict(best_state)
    return model, history


# ==============================================================================
# 7. Create Routing label
# ==============================================================================
def generate_routing_labels(model, X, y, batch_size=128, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X, y),
        batch_size=batch_size,
        shuffle=False
    )

    model.eval()
    model.to(device)

    all_y = []
    all_fob_pred = []
    all_base_pred = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            fob_logits, base_logits = model(xb)

            fob_pred = fob_logits.argmax(dim=1)
            base_pred = base_logits.argmax(dim=1)

            all_y.append(yb.cpu().numpy())
            all_fob_pred.append(fob_pred.cpu().numpy())
            all_base_pred.append(base_pred.cpu().numpy())

    y_true = np.concatenate(all_y)
    fob_pred = np.concatenate(all_fob_pred)
    base_pred = np.concatenate(all_base_pred)

    fob_correct = (fob_pred == y_true)
    base_correct = (base_pred == y_true)

    # 기본은 FOB(label=1)
    routing_labels = np.ones_like(y_true, dtype=np.int64)

    # FOB는 틀리고 Baseline만 맞는 경우만 baseline(label=2)
    routing_labels[(~fob_correct) & (base_correct)] = 2

    print("\n[Routing Label Generation]")
    print(f"Total samples              : {len(y_true)}")
    print(f"FOB correct                : {fob_correct.sum()} ({fob_correct.mean()*100:.2f}%)")
    print(f"Baseline correct           : {base_correct.sum()} ({base_correct.mean()*100:.2f}%)")
    print(f"Both correct               : {(fob_correct & base_correct).sum()}")
    print(f"FOB only correct           : {(fob_correct & ~base_correct).sum()}")
    print(f"Baseline only correct      : {((~fob_correct) & base_correct).sum()}")
    print(f"Both wrong                 : {((~fob_correct) & (~base_correct)).sum()}")
    print(f"Routing label 1 = FOB      : {(routing_labels == 1).sum()} ({(routing_labels == 1).mean()*100:.2f}%)")
    print(f"Routing label 2 = Baseline : {(routing_labels == 2).sum()} ({(routing_labels == 2).mean()*100:.2f}%)")

    return routing_labels, fob_pred, base_pred, y_true


# ==============================================================================
# 8. Train Decision Tree
# ==============================================================================
def train_decision_tree_obp(
    stat_train,
    routing_train,
    stat_test=None,
    routing_test=None,
    max_depth=None,
    min_samples_leaf=1,
    random_state=42
):
    dt = DecisionTreeClassifier(
        criterion="gini",
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state
    )

    dt.fit(stat_train, routing_train)

    train_pred = dt.predict(stat_train)

    print("\n[Decision Tree OBP - Train]")
    print(f"Accuracy : {accuracy_score(routing_train, train_pred):.4f}")
    print(f"Macro-F1 : {f1_score(routing_train, train_pred, average='macro'):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(routing_train, train_pred, labels=[1, 2]))

    if stat_test is not None and routing_test is not None:
        test_pred = dt.predict(stat_test)

        print("\n[Decision Tree OBP - Test]")
        print(f"Accuracy : {accuracy_score(routing_test, test_pred):.4f}")
        print(f"Macro-F1 : {f1_score(routing_test, test_pred, average='macro'):.4f}")
        print("Confusion Matrix:")
        print(confusion_matrix(routing_test, test_pred, labels=[1, 2]))

        print("\nClassification Report:")
        print(classification_report(
            routing_test,
            test_pred,
            labels=[1, 2],
            target_names=["FOB", "Baseline"],
            zero_division=0
        ))

    return dt


# ==============================================================================
# 9. Inference
# ==============================================================================
def adaptive_inference(
    model,
    dt_obp,
    X,
    stat_features,
    y,
    batch_size=128,
    device=None
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if isinstance(X, np.ndarray):
        X = torch.FloatTensor(X)
    if isinstance(y, np.ndarray):
        y = torch.LongTensor(y)

    model.eval()
    model.to(device)

    # DT routing prediction
    route_pred = dt_obp.predict(stat_features)

    final_preds = np.zeros(len(y), dtype=np.int64)
    fob_preds = np.zeros(len(y), dtype=np.int64)
    base_preds = np.zeros(len(y), dtype=np.int64)

    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            end = start + batch_size

            xb = X[start:end].to(device)

            fob_logits, base_logits = model(xb)

            fob_batch_pred = fob_logits.argmax(dim=1).cpu().numpy()
            base_batch_pred = base_logits.argmax(dim=1).cpu().numpy()

            fob_preds[start:end] = fob_batch_pred
            base_preds[start:end] = base_batch_pred

            routes = route_pred[start:end]

            # route=1이면 FOB 결과, route=2이면 Baseline 결과 사용
            final_preds[start:end] = np.where(
                routes == 1,
                fob_batch_pred,
                base_batch_pred
            )

    y_np = y.cpu().numpy() if isinstance(y, torch.Tensor) else y

    fob_f1 = f1_score(y_np, fob_preds, average="macro")
    base_f1 = f1_score(y_np, base_preds, average="macro")
    adaptive_f1 = f1_score(y_np, final_preds, average="macro")

    fob_ratio = np.mean(route_pred == 1)
    base_ratio = np.mean(route_pred == 2)

    print("\n[Adaptive Inference Result]")
    print(f"FOB Macro-F1      : {fob_f1:.4f}")
    print(f"Baseline Macro-F1 : {base_f1:.4f}")
    print(f"Proposed Macro-F1 : {adaptive_f1:.4f}")
    print(f"FOB selected      : {fob_ratio * 100:.2f}%")
    print(f"Baseline selected : {base_ratio * 100:.2f}%")

    return {
        "fob_f1": fob_f1,
        "baseline_f1": base_f1,
        "adaptive_f1": adaptive_f1,
        "route_pred": route_pred,
        "final_preds": final_preds,
        "fob_preds": fob_preds,
        "base_preds": base_preds,
        "fob_ratio": fob_ratio,
        "base_ratio": base_ratio,
    }


# ==============================================================================
# 10. Main
# ==============================================================================
def main():
    data_dir = "/content/drive/MyDrive/Colab Notebooks/datasets/UCI-HAR"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # 1. Load
    train_dataset = UCIHARDataset(data_dir, split="train")
    test_dataset = UCIHARDataset(data_dir, split="test")

    X_train_raw = train_dataset.X
    y_train = train_dataset.y
    X_test_raw = test_dataset.X
    y_test = test_dataset.y

    print("Raw train:", X_train_raw.shape)
    print("Raw test :", X_test_raw.shape)

    # 2. Downsampling
    X_train_down = downsample_interpolation(X_train_raw, target_len=32)
    X_test_down = downsample_interpolation(X_test_raw, target_len=32)

    # 3. Statistical features for DT
    stat_train = extract_ahar_stat_features(X_train_down)
    stat_test = extract_ahar_stat_features(X_test_down)

    # 4. Z-score normalization for CNN
    X_train_cnn = segment_wise_zscore(X_train_down)
    X_test_cnn = segment_wise_zscore(X_test_down)

    print("CNN train:", X_train_cnn.shape)
    print("Stat train:", stat_train.shape)

    # 5. Multi-output CNN
    model = MultiOutputCNN(in_channels=9, num_classes=6)

    # 6. CNN training
    model, history = train_multioutput_cnn(
        model=model,
        X_train=X_train_cnn,
        y_train=y_train,
        X_test=X_test_cnn,
        y_test=y_test,
        batch_size=64,
        epochs=100,
        lr=1e-3,
        weight_decay=0.0,
        device=device
    )

    # 7. Routing label generation
    routing_train, _, _, _ = generate_routing_labels(
        model=model,
        X=X_train_cnn,
        y=y_train,
        batch_size=128,
        device=device
    )

    routing_test, _, _, _ = generate_routing_labels(
        model=model,
        X=X_test_cnn,
        y=y_test,
        batch_size=128,
        device=device
    )

    # 8. DT training
    dt_obp = train_decision_tree_obp(
        stat_train=stat_train,
        routing_train=routing_train,
        stat_test=stat_test,
        routing_test=routing_test,
        max_depth=4,
        min_samples_leaf=10,
        random_state=42
    )

    # 9. Adaptive inference
    results = adaptive_inference(
        model=model,
        dt_obp=dt_obp,
        X=X_test_cnn,
        stat_features=stat_test,
        y=y_test,
        batch_size=128,
        device=device
    )

    return model, dt_obp, history, results


if __name__ == "__main__":
    model, dt_obp, history, results = main()

Device: cuda
Raw train: torch.Size([7352, 9, 128])
Raw test : torch.Size([2947, 9, 128])
CNN train: torch.Size([7352, 9, 32])
Stat train: (7352, 4)
[Epoch 001/100] Train Loss=3.5456 | Train FOB F1=0.2091, Base F1=0.1713 || Test FOB F1=0.3269, Base F1=0.2632 BEST
[Epoch 025/100] Train Loss=0.9603 | Train FOB F1=0.8132, Base F1=0.8138 || Test FOB F1=0.7608, Base F1=0.7589
[Epoch 050/100] Train Loss=0.8735 | Train FOB F1=0.8253, Base F1=0.8349 || Test FOB F1=0.7780, Base F1=0.7852
[Epoch 075/100] Train Loss=0.8531 | Train FOB F1=0.8274, Base F1=0.8379 || Test FOB F1=0.7800, Base F1=0.7869
[Epoch 100/100] Train Loss=0.8550 | Train FOB F1=0.8267, Base F1=0.8391 || Test FOB F1=0.7769, Base F1=0.7783

[Routing Label Generation]
Total samples              : 7352
FOB correct                : 6079 (82.68%)
Baseline correct           : 6228 (84.71%)
Both correct               : 5894
FOB only correct           : 185
Baseline only correct      : 334
Both wrong                 : 939
Routing label 1 